# Ribo-seq analysis from global BAM + Zarr

**Load once, query instantly.** Reads the entire BAM and GTF into memory,
does a single overlap join, then all downstream analyses (gene profiles,
count matrices, metagenes, clustering, range scoring) are in-memory operations.

Dependencies: `pip install biobear pyranges1 polars zarr matplotlib scikit-learn`

In [1]:
!which python     

/homes/jackt/micromamba/envs/riboseq-analysis/bin/python


In [2]:
# === EDIT THESE PATHS ===
BAM_PATH = "/hps/nobackup/flicek/ensembl/genebuild/jackt/riboseq/results/global/unique_reads.bam"
ZARR_PATH = "/hps/nobackup/flicek/ensembl/genebuild/jackt/riboseq/results/global/global_matrix.zarr"
GTF_PATH = "/hps/nobackup/flicek/ensembl/genebuild/jackt/riboseq/zebrafish_references/zebrafish_grcz12tu/GRCz12tu/danio_rerio_gca049306965v1.gtf"  # Ensembl GTF used for STAR index

# Optional: P-site offset file from RiboWaltz (tab-delimited: length\toffset)
OFFSET_FILE = None  # e.g., "/path/to/psite_offsets.tsv"

# Gene to examine
GENE_NAME = "ACTB"  # or use GENE_ID below
GENE_ID = "ENSDARG00160006181"       # e.g., "ENSG00000075624" — takes priority over GENE_NAME if set

# Samples to compare (None = all samples)
SAMPLES = None  # e.g., ["SRR1234567", "SRR7654321"]

In [4]:
import numpy as np
import polars as pl
import zarr
import pyranges1 as pr
import matplotlib.pyplot as plt
import time
from pathlib import Path
from collections import defaultdict

try:
    import biobear as bb
    HAS_BIOBEAR = True
except ImportError:
    import pysam
    HAS_BIOBEAR = False
    print("biobear not found — falling back to pysam (slower)")

## Helpers

In [5]:
# ── GTF parsing ─────────────────────────────────────────────────────

def _parse_attrs(s):
    d = {}
    for f in s.strip().split(';'):
        f = f.strip()
        if not f: continue
        k, _, v = f.partition(' ')
        d[k] = v.strip('"')
    return d


def parse_gtf(gtf_path):
    """
    Parse GTF into two polars DataFrames:
      genes_pl:    gene_id, gene_name, Chromosome, Start, End, Strand
      features_pl: gene_id, feature, Chromosome, Start, End, Strand
    """
    genes, feats = [], []
    with open(gtf_path) as f:
        for line in f:
            if line.startswith('#'): continue
            c = line.strip().split('\t')
            if len(c) < 9: continue
            feat = c[2]
            a = _parse_attrs(c[8])
            s, e = int(c[3]) - 1, int(c[4])  # 0-based half-open
            if feat == 'gene':
                genes.append((a.get('gene_id',''), a.get('gene_name',''),
                              c[0], s, e, c[6]))
            elif feat in ('exon', 'CDS'):
                feats.append((a.get('gene_id',''), feat, c[0], s, e, c[6]))
    genes_pl = pl.DataFrame(
        genes, schema=['gene_id','gene_name','Chromosome','Start','End','Strand'],
        orient='row')
    features_pl = pl.DataFrame(
        feats, schema=['gene_id','feature','Chromosome','Start','End','Strand'],
        orient='row')
    return genes_pl, features_pl


# ── BAM reading ────────────────────────────────────────────────────

def read_bam_biobear(bam_path):
    session = bb.connect()
    session.sql(f"""
        CREATE EXTERNAL TABLE _bam STORED AS BAM LOCATION '{bam_path}'
    """)
    raw = session.sql("""
        SELECT name, flag, reference, start, "end" FROM _bam
    """).to_polars()
    return (
        raw.filter(
            (pl.col('flag').and_(4) == 0) &    # mapped
            (pl.col('flag').and_(256) == 0) &   # primary
            (pl.col('flag').and_(2048) == 0)     # not supplementary
        )
        .filter(pl.col('name').str.starts_with('read_'))
        .with_columns(
            pl.col('name').str.strip_prefix('read_').cast(pl.Int64).alias('global_id'),
            pl.when(pl.col('flag').and_(16) > 0)
              .then(pl.lit('-')).otherwise(pl.lit('+')).alias('Strand'),
        )
        .rename({'reference': 'Chromosome', 'start': 'Start', 'end': 'End'})
        .select(['global_id', 'Chromosome', 'Start', 'End', 'Strand'])
    )


def read_bam_pysam(bam_path):
    """Fallback: single sequential pass with pysam."""
    rows = []
    with pysam.AlignmentFile(bam_path, 'rb') as bam:
        for r in bam.fetch(until_eof=True):
            if r.is_unmapped or r.is_secondary or r.is_supplementary:
                continue
            if not r.query_name.startswith('read_'):
                continue
            rows.append((
                int(r.query_name.split('_')[1]),
                r.reference_name,
                r.reference_start,
                r.reference_end,
                '-' if r.is_reverse else '+',
            ))
    return pl.DataFrame(
        rows, schema=['global_id','Chromosome','Start','End','Strand'],
        orient='row')


# ── Offsets ─────────────────────────────────────────────────────────

def load_offsets(path):
    d = {}
    with open(path) as f:
        next(f)
        for line in f:
            l, o = line.strip().split('\t')
            d[int(l)] = int(o)
    return d


# ── Plotting ────────────────────────────────────────────────────────

def plot_profiles(profiles, tx_length, cds_start, cds_end,
                  sample_names, title='', max_samples=6, normalise=False):
    n = min(len(sample_names), max_samples)
    ratios = [3] * n + [0.5]
    fig, axes = plt.subplots(
        n + 1, 1, figsize=(14, 2.2 * n + 1),
        gridspec_kw=dict(height_ratios=ratios, hspace=0.12), sharex=True)
    x = np.arange(tx_length)
    for i in range(n):
        ax = axes[i]
        y = profiles[i].copy()
        if normalise and y.sum() > 0:
            y = y / y.sum() * 1e6
        ax.bar(x, y, width=1.0, color='#2c7fb8', edgecolor='none', alpha=0.85)
        ax.axvspan(cds_start, cds_end, alpha=0.06, color='green', zorder=0)
        if cds_start > 0:
            ax.axvline(cds_start, color='green', lw=0.8, ls='--', alpha=0.5)
        if cds_end < tx_length:
            ax.axvline(cds_end, color='red', lw=0.8, ls='--', alpha=0.5)
        ax.set_ylabel('RPM' if normalise else 'Count', fontsize=9)
        ax.set_title(sample_names[i], fontsize=10, loc='left', pad=2)
        ax.spines[['top','right']].set_visible(False)
    # annotation track
    a = axes[-1]
    a.plot([0, tx_length], [.5,.5], color='#333', lw=2, solid_capstyle='butt')
    if cds_end > cds_start:
        a.add_patch(plt.Rectangle((cds_start,.2), cds_end-cds_start, .6,
                                  fc='#2c7fb8', ec='#333', lw=1, zorder=3))
        a.text((cds_start+cds_end)/2, .5, 'CDS', ha='center', va='center',
               fontsize=9, fontweight='bold', color='white', zorder=4)
    if cds_start > 0:
        a.text(cds_start/2, .5, "5'UTR", ha='center', va='center', fontsize=8, color='#555')
    if cds_end < tx_length:
        a.text((cds_end+tx_length)/2, .5, "3'UTR", ha='center', va='center', fontsize=8, color='#555')
    a.set_xlim(0, tx_length); a.set_ylim(0,1); a.axis('off')
    axes[-2].set_xlabel("Transcript position (nt, 5'→3')", fontsize=11)
    if title: fig.suptitle(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    return fig


print('Helpers loaded ✓')

Helpers loaded ✓


---
## Load everything (run once)

In [6]:
t0 = time.time()

# ── A. BAM → polars ────────────────────────────────────────────────
print('Reading BAM...', flush=True)
reads_pl = read_bam_biobear(BAM_PATH) if HAS_BIOBEAR else read_bam_pysam(BAM_PATH)
print(f'  {len(reads_pl):,} aligned reads  [{time.time()-t0:.0f}s]')

# ── B. GTF → polars ────────────────────────────────────────────────
print('Parsing GTF...', flush=True)
genes_pl, features_pl = parse_gtf(GTF_PATH)
print(f'  {len(genes_pl):,} genes, {len(features_pl):,} exon/CDS features')

# ── C. Overlap join ────────────────────────────────────────────────
print('Overlap join (reads × genes)...', flush=True)
reads_pr = pr.PyRanges(
    reads_pl.select(['Chromosome','Start','End','Strand','global_id']).to_pandas())
genes_pr = pr.PyRanges(genes_pl.to_pandas())
overlap_pd = genes_pr.join_ranges(reads_pr, strand_behavior='same').df
overlap_df = pl.from_pandas(overlap_pd)
print(f'  {len(overlap_df):,} read-gene overlaps  [{time.time()-t0:.0f}s]')

# ── D. Zarr + samples ──────────────────────────────────────────────
root = zarr.open(ZARR_PATH, mode='r')
counts = root['counts']
sample_names = list(root.attrs['samples'])
chunk_size = counts.chunks[0]

if SAMPLES:
    sample_idx = np.array([sample_names.index(s) for s in SAMPLES])
    sel_samples = SAMPLES
else:
    sample_idx = np.arange(len(sample_names))
    sel_samples = sample_names

n_samples = len(sel_samples)
print(f'  Zarr: {counts.shape[0]:,} reads × {counts.shape[1]:,} samples')
print(f'  Selected {n_samples} samples')
print(f'\n✓ Ready  [{time.time()-t0:.0f}s total]')

Reading BAM...
  22,677,115 aligned reads  [24s]
Parsing GTF...
  54,881 genes, 850,951 exon/CDS features
Overlap join (reads × genes)...


AttributeError: 'PyRanges' object has no attribute 'join_ranges'

---
## Gene × sample count matrix (single zarr pass)

In [ ]:
t0 = time.time()

# Group overlaps: gene_id → sorted list of global_ids
gene_reads_map = (
    overlap_df
    .select(['gene_id', 'global_id'])
    .unique()
    .sort('global_id')
    .group_by('gene_id')
    .agg(pl.col('global_id'))
)

gene_ids_list = gene_reads_map['gene_id'].to_list()
gene_id_to_idx = {g: i for i, g in enumerate(gene_ids_list)}
n_genes = len(gene_ids_list)

# Reverse index: global_id → list of gene matrix row indices
gid_to_gene_rows = defaultdict(list)
for row in gene_reads_map.iter_rows():
    gene_id, gids = row
    gi = gene_id_to_idx[gene_id]
    for gid in gids:
        gid_to_gene_rows[gid].append(gi)

needed_ids = np.array(sorted(gid_to_gene_rows.keys()), dtype=np.int64)
print(f'{n_genes:,} genes with reads, {len(needed_ids):,} unique read ids')

# Single sequential pass through zarr chunks
result = np.zeros((n_genes, n_samples), dtype=np.uint64)
n_chunks_read = 0

for chunk_start in range(0, counts.shape[0], chunk_size):
    chunk_end = min(chunk_start + chunk_size, counts.shape[0])
    lo = np.searchsorted(needed_ids, chunk_start, side='left')
    hi = np.searchsorted(needed_ids, chunk_end, side='left')
    if lo >= hi:
        continue

    chunk = counts[chunk_start:chunk_end, :]  # read once
    n_chunks_read += 1

    for idx in range(lo, hi):
        gid = int(needed_ids[idx])
        row_in_chunk = gid - chunk_start
        rc = chunk[row_in_chunk, sample_idx].astype(np.uint64)
        for gene_row in gid_to_gene_rows[gid]:
            result[gene_row, :] += rc

    if n_chunks_read % 200 == 0:
        print(f'  {n_chunks_read} chunks read  [{time.time()-t0:.0f}s]', flush=True)

elapsed = time.time() - t0
print(f'\n✓ {n_genes:,} genes × {n_samples} samples  [{elapsed:.0f}s, {n_chunks_read} chunks]')

# Attach gene names
gene_name_map = dict(zip(
    genes_pl['gene_id'].to_list(), genes_pl['gene_name'].to_list()))
gene_names_list = [gene_name_map.get(g, '') for g in gene_ids_list]

import pandas as pd
gene_count_df = pd.DataFrame(
    result.astype(np.uint32),
    index=pd.MultiIndex.from_arrays(
        [gene_ids_list, gene_names_list], names=['gene_id','gene_name']),
    columns=sel_samples,
)
gene_count_df.head(10)

In [ ]:
out = str(Path(ZARR_PATH).parent / 'gene_counts_matrix.tsv')
gene_count_df.to_csv(out, sep='\t')
print(f'Saved to {out}')

---
## Single-gene profile (instant)

In [ ]:
# ── Pick gene ──────────────────────────────────────────────────────
if GENE_ID:
    _gf = genes_pl.filter(pl.col('gene_id') == GENE_ID)
else:
    _gf = genes_pl.filter(pl.col('gene_name') == GENE_NAME)
assert len(_gf) > 0, f"Gene not found: {GENE_ID or GENE_NAME}"
gene = _gf.row(0, named=True)

# ── Reads overlapping this gene (from pre-computed join) ──────────
gene_overlap = overlap_df.filter(pl.col('gene_id') == gene['gene_id'])
gene_gids = gene_overlap['global_id'].unique().sort().to_list()
gene_reads = reads_pl.filter(pl.col('global_id').is_in(gene_gids))

# ── Build transcript coord map ────────────────────────────────────
gf = features_pl.filter(pl.col('gene_id') == gene['gene_id'])
exons = [(r[2], r[3]) for r in gf.filter(pl.col('feature')=='exon').select(['Start','End']).iter_rows()]
cdss  = [(r[0], r[1]) for r in gf.filter(pl.col('feature')=='CDS').select(['Start','End']).iter_rows()]
exons = sorted(set(exons)); cdss = sorted(set(cdss))

exon_pos = sorted({p for s,e in exons for p in range(s,e)})
if gene['Strand'] == '-': exon_pos = exon_pos[::-1]
g2tx = {gp: tx for tx, gp in enumerate(exon_pos)}
tx_len = len(exon_pos)

cds_genomic = {p for s,e in cdss for p in range(s,e)}
cds_tx = sorted(g2tx[g] for g in cds_genomic if g in g2tx)
cds_s = cds_tx[0] if cds_tx else 0
cds_e = cds_tx[-1]+1 if cds_tx else 0

# ── Inflate from zarr ──────────────────────────────────────────────
gids_arr = np.array(gene_gids, dtype=np.int64)
id_to_counts = {}
by_chunk = defaultdict(list)
for gid in gids_arr:
    by_chunk[gid // chunk_size].append(gid)
for ci in sorted(by_chunk):
    cs, ce = ci*chunk_size, min((ci+1)*chunk_size, counts.shape[0])
    ch = counts[cs:ce, :]
    for gid in by_chunk[ci]:
        id_to_counts[gid] = ch[gid-cs, sample_idx]

# ── Build profile ──────────────────────────────────────────────────
offsets = load_offsets(OFFSET_FILE) if OFFSET_FILE else None
profiles = np.zeros((n_samples, tx_len), dtype=np.float64)

for row in gene_reads.iter_rows(named=True):
    sc = id_to_counts.get(row['global_id'])
    if sc is None: continue
    alen = row['End'] - row['Start']  # contiguous (no introns, alignIntronMax=1)

    if offsets:
        ofs = offsets.get(alen)
        if ofs is None or ofs >= alen: continue
        if row['Strand'] == '+':
            asite = row['Start'] + ofs
        else:
            asite = row['End'] - 1 - ofs
        ti = g2tx.get(asite)
        if ti is not None:
            profiles[:, ti] += sc
    else:
        for p in range(row['Start'], row['End']):
            ti = g2tx.get(p)
            if ti is not None:
                profiles[:, ti] += sc

print(f"{gene['gene_name']} ({gene['gene_id']})")
print(f"  {gene['Chromosome']}:{gene['Start']:,}-{gene['End']:,} ({gene['Strand']})")
print(f"  {len(gene_gids)} unique reads | tx_len={tx_len} | CDS={cds_s}-{cds_e}")
print(f"  Mode: {'A-site' if offsets else 'coverage'}")

In [ ]:
plot_profiles(profiles, tx_len, cds_s, cds_e, sel_samples,
    title=f"{gene['gene_name']} ({gene['gene_id']})");

In [ ]:
plot_profiles(profiles, tx_len, cds_s, cds_e, sel_samples,
    title=f"{gene['gene_name']} — normalised", normalise=True);

---
## Genome-wide metagene

In [ ]:
N_BINS = 100
t0 = time.time()

# Get CDS features per gene
cds_features = features_pl.filter(pl.col('feature') == 'CDS')
cds_by_gene = (
    cds_features
    .group_by('gene_id')
    .agg([pl.col('Start').min().alias('cds_min'),
          pl.col('End').max().alias('cds_max'),
          (pl.col('End') - pl.col('Start')).sum().alias('cds_length')])
    .filter(pl.col('cds_length') >= 90)  # skip very short CDS
)

# For each gene with CDS and reads, build binned CDS profile
meta_acc = np.zeros((n_samples, N_BINS), dtype=np.float64)
n_genes_used = 0

# Pre-filter overlaps to CDS genes only
cds_gene_ids = set(cds_by_gene['gene_id'].to_list())
cds_overlaps = overlap_df.filter(pl.col('gene_id').is_in(list(cds_gene_ids)))

# Get per-read CDS-relative position:
# Join reads with CDS boundaries, compute where read falls in CDS
# For speed, use the gene count matrix result to skip genes with 0 reads
genes_with_reads = set(gene_ids_list)  # from the matrix computation
cds_gene_ids = cds_gene_ids & genes_with_reads

for gid in cds_gene_ids:
    gf_cds = cds_features.filter(pl.col('gene_id') == gid)
    gene_row = genes_pl.filter(pl.col('gene_id') == gid)
    if len(gene_row) == 0: continue
    strand = gene_row['Strand'][0]

    # CDS positions in reading order
    cds_pos = []
    for r in gf_cds.iter_rows(named=True):
        cds_pos.extend(range(r['Start'], r['End']))
    cds_pos = sorted(set(cds_pos))
    if strand == '-': cds_pos = cds_pos[::-1]
    cds_len = len(cds_pos)
    if cds_len < 90: continue

    pos_to_frac = {p: i/cds_len for i, p in enumerate(cds_pos)}

    # Get reads for this gene
    if gid not in gene_id_to_idx: continue
    gi = gene_id_to_idx[gid]
    gene_total = result[gi, :]  # already computed
    if gene_total.sum() == 0: continue

    # Get read-level detail from overlap + reads_pl
    g_overlaps = cds_overlaps.filter(pl.col('gene_id') == gid)
    g_gids = g_overlaps['global_id'].unique().to_list()
    g_reads = reads_pl.filter(pl.col('global_id').is_in(g_gids))

    # Build per-position profile for this gene (summed across all samples for binning)
    gene_profile = np.zeros((n_samples, cds_len), dtype=np.float64)
    for rd in g_reads.iter_rows(named=True):
        sc = id_to_counts.get(rd['global_id'])
        # Use the preloaded id_to_counts if available, otherwise skip
        # For metagene we need ALL reads — inflate on the fly
        if sc is None:
            # Quick zarr lookup
            ci = rd['global_id'] // chunk_size
            cs = ci * chunk_size
            ce = min(cs + chunk_size, counts.shape[0])
            ch = counts[cs:ce, :]
            sc = ch[rd['global_id'] - cs, sample_idx]
        for p in range(rd['Start'], rd['End']):
            if p in pos_to_frac:
                cidx = int(pos_to_frac[p] * cds_len)
                if 0 <= cidx < cds_len:
                    gene_profile[:, cidx] += sc

    # Bin into N_BINS and normalise per gene
    binned = np.zeros((n_samples, N_BINS), dtype=np.float64)
    for b in range(N_BINS):
        si = int(b * cds_len / N_BINS)
        ei = int((b+1) * cds_len / N_BINS)
        if ei > si:
            binned[:, b] = gene_profile[:, si:ei].sum(axis=1)

    # Normalise per-sample so each gene contributes equally
    for s in range(n_samples):
        total = binned[s].sum()
        if total > 0:
            meta_acc[s] += binned[s] / total

    n_genes_used += 1
    if n_genes_used % 500 == 0:
        print(f'  {n_genes_used} genes  [{time.time()-t0:.0f}s]', flush=True)

print(f'✓ Metagene from {n_genes_used} genes  [{time.time()-t0:.0f}s]')

# Plot
fig, ax = plt.subplots(figsize=(12, 3.5))
x = np.linspace(0, 100, N_BINS)
for i in range(min(6, n_samples)):
    y = meta_acc[i]
    if y.sum() > 0: y = y / y.sum() * 100
    ax.plot(x, y, label=sel_samples[i], alpha=0.7, lw=1)
ax.axvline(0, color='green', ls='--', alpha=0.3)
ax.axvline(100, color='red', ls='--', alpha=0.3)
ax.set(xlabel='CDS position (%)', ylabel='Relative coverage (%)',
       title=f'Genome-wide CDS metagene ({n_genes_used} genes)')
ax.legend(fontsize=7, ncol=2)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout();

---
## Sample clustering

In [ ]:
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import pdist

# Log-normalise the gene count matrix
mat = gene_count_df.values.astype(np.float64).T  # (samples × genes)
lib_sizes = mat.sum(axis=1, keepdims=True).clip(1)
log_cpm = np.log2(mat / lib_sizes * 1e6 + 1)

# PCA
pca = PCA(n_components=min(20, *log_cpm.shape))
pcs = pca.fit_transform(log_cpm)

# K-means (auto-pick k via elbow heuristic, or use k=5)
k = min(5, n_samples // 2) if n_samples >= 10 else 2
km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(pcs[:, :5])

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# PCA scatter
ax = axes[0]
scatter = ax.scatter(pcs[:,0], pcs[:,1], c=km.labels_, cmap='tab10',
                     s=30, alpha=0.7, edgecolors='black', linewidths=0.3)
ax.set(xlabel=f'PC1 ({pca.explained_variance_ratio_[0]:.1%})',
       ylabel=f'PC2 ({pca.explained_variance_ratio_[1]:.1%})',
       title=f'PCA — {n_samples} samples, k={k}')
ax.spines[['top','right']].set_visible(False)

# Correlation heatmap
corr = np.corrcoef(log_cpm)
# Reorder by hierarchical clustering
Z = linkage(pdist(log_cpm, metric='correlation'), method='average')
order = leaves_list(Z)
corr_ordered = corr[np.ix_(order, order)]

ax = axes[1]
im = ax.imshow(corr_ordered, cmap='RdYlBu_r', vmin=0, vmax=1, aspect='auto')
ax.set_title(f'Sample correlation ({n_samples} samples)')
ax.set_xticks([]); ax.set_yticks([])
plt.colorbar(im, ax=ax, label='Pearson r', shrink=0.8)

plt.tight_layout();
print(f'Variance explained (top 5 PCs): {pca.explained_variance_ratio_[:5].sum():.1%}')

---
## Score arbitrary ranges

In [ ]:
def score_ranges(ranges_df, reads_pl, counts, sample_idx, chunk_size):
    """
    Count reads per sample for arbitrary genomic ranges.

    Args:
        ranges_df: polars DataFrame with Chromosome, Start, End, Strand, range_id
        reads_pl:  pre-loaded reads DataFrame
        counts:    zarr counts array
        sample_idx: sample column indices
        chunk_size: zarr chunk size

    Returns:
        result: np.array (n_ranges, n_samples) of raw counts
    """
    # Overlap join
    ranges_pr = pr.PyRanges(ranges_df.to_pandas())
    reads_pr = pr.PyRanges(
        reads_pl.select(['Chromosome','Start','End','Strand','global_id']).to_pandas())
    joined = pl.from_pandas(
        ranges_pr.join_ranges(reads_pr, strand_behavior='same').df)

    if len(joined) == 0:
        return np.zeros((len(ranges_df), len(sample_idx)), dtype=np.uint64)

    # Group: range_id → global_ids
    range_ids = ranges_df['range_id'].to_list()
    rid_to_idx = {r: i for i, r in enumerate(range_ids)}

    gid_to_range_rows = defaultdict(list)
    pairs = joined.select(['range_id','global_id']).unique()
    for r in pairs.iter_rows():
        ri = rid_to_idx.get(r[0])
        if ri is not None:
            gid_to_range_rows[r[1]].append(ri)

    needed = np.array(sorted(gid_to_range_rows.keys()), dtype=np.int64)
    n_ranges = len(range_ids)
    n_s = len(sample_idx)
    result = np.zeros((n_ranges, n_s), dtype=np.uint64)

    for cs in range(0, counts.shape[0], chunk_size):
        ce = min(cs + chunk_size, counts.shape[0])
        lo = np.searchsorted(needed, cs, side='left')
        hi = np.searchsorted(needed, ce, side='left')
        if lo >= hi: continue
        chunk = counts[cs:ce, :]
        for idx in range(lo, hi):
            gid = int(needed[idx])
            rc = chunk[gid - cs, sample_idx].astype(np.uint64)
            for ri in gid_to_range_rows[gid]:
                result[ri, :] += rc

    return result


# ── Example: score a few ranges ────────────────────────────────────
# Use the first 5 genes as demo ranges
demo_ranges = (
    genes_pl.head(5)
    .with_columns(pl.col('gene_id').alias('range_id'))
    .select(['range_id', 'Chromosome', 'Start', 'End', 'Strand'])
)

scores = score_ranges(demo_ranges, reads_pl, counts, sample_idx, chunk_size)

score_df = pd.DataFrame(
    scores, index=demo_ranges['range_id'].to_list(), columns=sel_samples)
print(f'Range scores: {score_df.shape}')
score_df

---
## Reading frame (single gene)

In [ ]:
if cds_e > cds_s:
    cds_prof = profiles[:, cds_s:cds_e]
    fc = np.zeros((n_samples, 3))
    for f in range(3):
        fc[:, f] = cds_prof[:, f::3].sum(axis=1)

    n_plots = min(4, n_samples)
    fig, axes = plt.subplots(1, n_plots, figsize=(3.5*n_plots, 3))
    if n_plots == 1: axes = [axes]
    colors = ['#2ecc71','#f39c12','#e74c3c']
    for i, ax in enumerate(axes):
        tot = fc[i].sum()
        pct = fc[i]/tot*100 if tot > 0 else fc[i]
        ax.bar([0,1,2], pct, color=colors, ec='black', lw=0.5)
        ax.set(xticks=[0,1,2], xticklabels=['F0','F1','F2'],
               ylabel='% CDS reads', ylim=(0,100), title=sel_samples[i])
        ax.spines[['top','right']].set_visible(False)
    fig.suptitle(f"{gene['gene_name']} — Reading Frame", fontweight='bold')
    plt.tight_layout()
else:
    print('No CDS — skipping frame analysis')